# เฉลย Lab 08 — Customer Churn Prediction (ฉบับสมบูรณ์)

> นี่คือ "ฉบับอาจารย์" — ฉบับของนักศึกษาต้องมี EDA/error analysis/สไลด์ของตัวเอง
> โครงและตัวเลขอ้างอิงสามารถเทียบได้เมื่อใช้ random_state เดียวกัน

In [1]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    df = pd.read_csv(URL)
    print("loaded from URL:", df.shape)
except Exception as e:
    print("URL failed:", e, "→ synthetic fallback (see starter notebook)")
    from sklearn.datasets import make_classification
    Xs_, ys_ = make_classification(n_samples=3000, n_features=15, n_informative=8,
                                   weights=[0.735, 0.265], random_state=42)
    df = pd.DataFrame(Xs_, columns=[f"feat_{i}" for i in range(15)])
    df["tenure"] = np.abs(Xs_[:, 0] * 20 + 30).round(0)
    df["MonthlyCharges"] = np.abs(Xs_[:, 1] * 30 + 70).round(2)
    df["TotalCharges"] = df["tenure"] * df["MonthlyCharges"]
    df["Contract"] = np.where(Xs_[:, 2] > 0, "Month-to-month", "Two year")
    df["InternetService"] = np.where(Xs_[:, 3] > 0.5, "Fiber optic", "DSL")
    df["Churn"] = np.where(ys_ == 1, "Yes", "No")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # ' ' → NaN (11 จุด)

loaded from URL: (7043, 21)


## Step 1 — EDA: 3 พาดหัวข่าว

1. **"เดือนต่อเดือนคือเขตอันตราย: ยกเลิก 42.7% เทียบสัญญา 2 ปีที่แค่ 2.8%"**
2. **"ลูกค้า fiber ยกเลิกมากผิดปกติ (41.9%) แพ้เฉพาะ..." (DSL 19.0%)**
3. **"คนที่ยกเลิกคือมือใหม่: อายุสมาชิกเฉลี่ย 18 เดือน vs อยู่ต่อ 38 เดือน — แถมจ่ายแพงกว่า ($74 vs $61)"**

In [2]:
churn = (df["Churn"] == "Yes").astype(int)
print("churn rate:", round(churn.mean(), 3))
print(df.assign(churn=churn).groupby("Contract")["churn"].agg(["mean", "count"]).round(3))
print(df.assign(churn=churn).groupby("InternetService")["churn"].mean().round(3))
print(df.assign(churn=churn).groupby("churn")[["tenure", "MonthlyCharges"]].mean().round(1))

churn rate: 0.265
                 mean  count
Contract                    
Month-to-month  0.427   3875
One year        0.113   1473
Two year        0.028   1695
InternetService
DSL            0.190
Fiber optic    0.419
No             0.074
Name: churn, dtype: float64
       tenure  MonthlyCharges
churn                        
0        37.6            61.3
1        18.0            74.4


## Step 2 — split ก่อน + pipeline

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["customerID", "Churn"])
y = churn
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])
print("cat cols:", len(cat_cols), "| ok")

cat cols: 16 | ok


## Step 3 — baseline + metric

เลือก **ROC-AUC เป็นตัวตัดสินหลัก** (threshold-free เหมาะกับการจัดอันดับความเสี่ยง)
และดู **recall** เป็นรอง (บริบท retention: พลาดลูกค้าที่กำลังจะยกเลิก = เสียรายได้)
accuracy โกหกเพราะ 73.5% ของลูกค้าอยู่ต่อ — ทายว่า "ไม่มีใครยกเลิก" ก็ได้ 73.5% ฟรี

In [4]:
from sklearn.dummy import DummyClassifier

skf = StratifiedKFold(5, shuffle=True, random_state=42)
dummy = Pipeline([("pre", pre), ("m", DummyClassifier(strategy="most_frequent"))])
print("Dummy CV acc:", round(cross_val_score(dummy, Xtr, ytr, cv=skf).mean(), 3),
      "| AUC:", round(cross_val_score(dummy, Xtr, ytr, cv=skf, scoring="roc_auc").mean(), 3))

Dummy CV acc: 0.735 | AUC: 0.5


## Step 4 — model zoo (CV, ยังไม่แตะ test)

ผลอ้างอิง: Dummy 0.500 · RF 0.818 · HistGB 0.838 · **LR 0.846** — Linear ธรรมดาชนะ!
(เหตุผล: ฟีเจอร์ categorical สะอาด + ความสัมพันธ์ส่วนใหญ่เป็นเชิงเส้น/monotone)

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

zoo = [
    ("RandomForest", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ("HistGradBoost", HistGradientBoostingClassifier(random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=2000)),
]
for name, m in zoo:
    pipe = Pipeline([("pre", pre), ("m", m)])
    auc = cross_val_score(pipe, Xtr, ytr, cv=skf, scoring="roc_auc").mean()
    acc = cross_val_score(pipe, Xtr, ytr, cv=skf).mean()
    print(f"{name:20s} CV acc={acc:.3f}  CV AUC={auc:.3f}")

RandomForest         CV acc=0.784  CV AUC=0.818


HistGradBoost        CV acc=0.793  CV AUC=0.838


LogisticRegression   CV acc=0.803  CV AUC=0.846


## Step 5 — tune LogisticRegression เบา ๆ

C คือ "ความกล้า" ของโมเดล (คู่ผกผันของแรง regularization) — ลองสุ่มหาค่าที่ดี

In [6]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(
    Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=2000))]),
    {"m__C": loguniform(0.01, 100)},
    n_iter=20, cv=skf, scoring="roc_auc", random_state=42)
rs.fit(Xtr, ytr)
print("best CV AUC:", round(rs.best_score_, 3), "| C:", round(rs.best_params_["m__C"], 3))

best = rs.best_estimator_

best CV AUC: 0.846 | C: 8.472


## Step 6 — threshold คือเรื่องงบประมาณ

In [7]:
from sklearn.metrics import precision_score, recall_score

proba_tr = best.predict_proba(Xtr)[:, 1]
for th in [0.35, 0.50, 0.65]:
    yp = (proba_tr >= th).astype(int)
    print(f"th={th:.2f}: recall={recall_score(ytr, yp):.2f}  precision={precision_score(ytr, yp):.2f}")

# ประโยค decision-ready (ตัวอย่าง):
print()
print('เกณฑ์ 0.5: จับ churner ได้ ~56% โทรถูกคน ~66% | หลั่งเป็น 0.35: จับได้ ~71% แต่โทรเปล่าเกือบครึ่ง')
print('งบพอโทร 100 คน/เดือน → เรียง predict_proba แล้วโทร top-100 ก่อนเสมอ')

th=0.35: recall=0.72  precision=0.58
th=0.50: recall=0.55  precision=0.66
th=0.65: recall=0.31  precision=0.76

เกณฑ์ 0.5: จับ churner ได้ ~56% โทรถูกคน ~66% | หลั่งเป็น 0.35: จับได้ ~71% แต่โทรเปล่าเกือบครึ่ง
งบพอโทร 100 คน/เดือน → เรียง predict_proba แล้วโทร top-100 ก่อนเสมอ


## Step 7 — drivers + top misses

tenure −1.23σ (อายุสมาชิกยิ่งนานยิ่งเกาะติด) · Contract_Two year −0.76 · Fiber +0.64 ·
Contract_Month-to-month +0.59 — ตรงเป๊ะกับ EDA ข้อ 1-2 (โมเดลกับตาเห็นตรงกัน = สัญญาณที่ดี)

In [8]:
import pandas as pd

feature_names = best[:-1].get_feature_names_out()
coefs = pd.Series(best[-1].coef_[0], index=[n.split("__")[1] for n in feature_names])
coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(2)

MonthlyCharges                       -1.93
InternetService_Fiber optic           1.28
tenure                               -1.28
InternetService_DSL                  -1.11
Contract_Two year                    -0.82
OnlineSecurity_No internet service   -0.57
InternetService_No                   -0.57
OnlineBackup_No internet service     -0.57
dtype: float64

In [9]:
# top misses บน train (churner ที่โมเดลมั่นใจว่าปลอดภัยที่สุด — กลุ่มอันตรายที่สุดที่หลุดไป)
train_miss = pd.DataFrame({"proba": proba_tr, "y": ytr.values})
train_miss[(train_miss.y == 1)].nsmallest(5, "proba")

,proba,y
4055,0.003902,1
4030,0.004002,1
3318,0.006005,1
5305,0.008288,1
1176,0.008465,1


**Insight สำหรับทีม retention (ตัวอย่าง):**
"ลูกค้า fiber แบบรายเดือนที่อายุสมาชิก < 12 เดือนคือกลุ่มเสี่ยงสูงสุด — โทรหาในเดือนที่ 3 (ก่อนหมดสัญญาปีแรก)
พร้อมเสนอส่วนลดหรือเปลี่ยนสัญญา 1 ปี โดยเรียงลำดับการโทรจาก predict_proba"

## Step 8 — คำพิพากษา (test ครั้งเดียว) + ส่งมอบ

In [10]:
import joblib
from sklearn.metrics import f1_score, roc_auc_score

proba = best.predict_proba(Xte)[:, 1]
yp = (proba >= 0.5).astype(int)
print(f"CV AUC 0.846 → test AUC {roc_auc_score(yte, proba):.3f}   ← gap จิ๋ว ๆ = ซื่อสัตย์")
print(f"recall {recall_score(yte, yp):.2f} | f1 {f1_score(yte, yp):.3f} | accuracy {(yte == yp).mean():.3f}")

joblib.dump(best, "churn_pipeline.joblib")

# พิสูจน์ว่าโหลดกลับมาใช้กับข้อมูลดิบได้
loaded = joblib.load("churn_pipeline.joblib")
loaded.predict(Xte[:3])

CV AUC 0.846 → test AUC 0.841   ← gap จิ๋ว ๆ = ซื่อสัตย์
recall 0.56 | f1 0.603 | accuracy 0.805


array([0, 1, 0])

## สรุปฉบับ present 5 นาที (โครงจริง)

- **Problem:** churn 26.5% × รายได้ต่อราย — retention ถูกกว่าหาลูกค้าใหม่ 5-25 เท่า
- **Method:** split-first pipeline (19 คอลัมน์ → one-hot + scale อัตโนมัติ)
- **Results:** baseline 0.500 → **LR test AUC 0.842** (CV 0.846 gap น้อย) — ชนะ forest/boosting เสียด้วยซ้ำ
- **Insight:** tenure + ประเภทสัญญา + fiber คือ 3 แรงหลัก → นโยบาย "โทรเดือนที่ 3"
- **Limits:** ข้อมูล snapshot ครั้งเดียว (ไม่มี time-series) · อาจมี leak จากฟีเจอร์ที่บันทึกหลังยกเลิก · ต้อง retrain ทุกไตรมาส